# Optical Flow

### Few Assumptions:
1. Pixel intensities of an object do not change between consecutive frames.
2. Neighbouring pixels have similar motion.

### Lucas-Kanade Function 
We will pass previous frame, previous points and the current frame to the function.

-> It tracks only the points that it is told to track.

In [1]:
import numpy as np
import cv2

In [6]:
# Tracking 10 corners and will try to detect those
corner_track_params = dict(maxCorners=10, qualityLevel=0.3, minDistance=7, blockSize=7)

In [7]:
# Defining Lucas-Kanade Parameter
lk_params = dict(winSize=(200,200), maxLevel=2, criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

In [8]:
# Capturing the live feed
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# Points to track by passing the previous corners points that we extracted
prevPts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **corner_track_params)

mask = np.zeros_like(prev_frame)

# Now continously tracking those points movements
while True:
    # Tracking the current frame
    ret, frame = cap.read() 
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Passing the previous frame, prevoius points and current frame in Lucas-Kanade function
    nextPts, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, prevPts, None, **lk_params)
    
    good_new = nextPts[status==1]
    good_prev = nextPts[status==1]

    for i, (new, prev) in enumerate(zip(good_new, good_prev)):
        x_new, y_new = new.ravel()
        x_prev, y_prev = prev.ravel()

        x_new, y_new = int(x_new), int(y_new)
        x_prev, y_prev = int(x_prev), int(y_prev)

        mask = cv2.line(mask, (x_new, y_new), (x_prev, y_prev), (0, 255, 0), 3)
        frame = cv2.circle(frame, (x_new, y_new), 8, (0, 0, 255), -1)

    img = cv2.add(frame, mask)
    cv2.imshow('tracking', img)

    k = cv2.waitKey(30) & 0xFF

    if k == 27:
        break

    prev_gray = frame_gray.copy()
    prevPts = good_new.reshape(-1,1,2)

cv2.destroyAllWindows()
cap.release()

# Dense Optical Flow

### calcOpticalFlowFarneback(prev, next, flow, pyr_scale, levels, winsize, iterations, poly_n, poly_sigma, flags)
Here are the parameters of the function and it's representation:
1. prev: First input frame (grayscale)
2. next: Second input frame (grayscale)
3. flow: Output flow (usually set to None)
4. pyr_scale: Image scale between pyramid levels, Value between 0 and 1.
5. levels: Number of pyramid layers
6. winsize: Averaging window size
7. iterations: Number of iterations per pyramid level
8. poly_n: Size of pixel neighborhood, Used for polynomial expansion.
9. poly_sigma: Standard deviation of Gaussian used in polynomial expansion.

In [12]:
cap = cv2.VideoCapture(0)
ret, frame1 = cap.read()
prvsImg = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

hsv_mask = np.zeros_like(frame1)
hsv_mask[:,:,1] = 255

while True:
    ret, frame2 = cap.read()
    nextImg = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prvsImg, nextImg, None, 0.5, 3, 15, 3, 4, 1.2, 0)

    mag, ang = cv2.cartToPolar(flow[:,:,0], flow[:,:,1], angleInDegrees=True)

    hsv_mask[:,:,0] = ang/2
    hsv_mask[:,:,2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

    bgr = cv2.cvtColor(hsv_mask, cv2.COLOR_HSV2BGR)
    cv2.imshow('frame', bgr)

    k = cv2.waitKey(10) & 0xFF
    if k == 27:
        break

    prevImg = nextImg

cap.release()
cv2.destroyAllWindows()